# Lesson 21 Lab — Quantizing Vision and Multimodal Models

**Puzzle:** Why can a text-only calibration set miss important failure modes in a vision-language model?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

A multimodal model does not have one activation distribution. Patch projection sees pixels and local contrast, the vision encoder sees image tokens, the connector maps modalities, and the language decoder sees text-conditioned states. Calibrating only text can leave the vision path with unobserved ranges and brittle low-bit behavior.


## 0. Predict before running

1. Predict how high-contrast image patches change the output error of one quantized patch projection.
2. Identify the calibration strata needed for a vision-language model rather than a text-only LLM.
3. Explain why a patch-projection result cannot establish full VLM quality.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

A vision-language system contains a vision encoder, patch/token embedding, projector, cross- or self-attention, language model, and KV cache. Each component sees a different activation distribution.

- Vision encoders see patch distributions, image contrast, and positional structure unlike text MLP activations.
- A multimodal pipeline contains encoder, projector, language model, and attention/cache objects.
- Coverage and fallback decisions can differ by component.


## 2. Derive the mechanism

Patch projection maps local pixel statistics into tokens; contrast and modality shifts can create channel ranges absent from text calibration. Quantization error can then propagate through normalization and attention.

A ViT patch projection is a convolution with kernel and stride equal to patch size. For 16×16 RGB patches, each output token combines 768 input values. Weight quantization error is filtered by the image distribution: high-contrast or sparse extreme pixels can amplify particular columns that ordinary Gaussian-like calibration does not emphasize.

Farther downstream, cross-attention and modality connectors introduce their own outliers and quality objectives. A sound plan therefore calibrates and evaluates per component and per modality slice, then rejoins them with end-to-end captioning, VQA, OCR, or grounding tasks.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "21-multimodal-quantization"
device = require_cuda()
torch.manual_seed(2026 + 21)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | floating-point 64-channel, 16×16 patch projection |
| Candidate | group-192 INT4-dequantized projection weights |
| Held constant | same weights, image shape, projection stride, normal/high-contrast paired inputs |
| Measurements | projection-output RMSE/MAE/cosine/max error for each image distribution |
| Evidence | `pytorch-gpu` |

**Experiment:** Quantize a CUDA patch-projection weight and compare reconstruction error for ordinary and high-contrast synthetic images.


## 5. Read the experiment code

The notebook isolates a patch projection and compares normal versus high-contrast image distributions, carefully avoiding a full-VLM claim.

The notebook isolates the first vision operation so tensor axes remain readable: weights have shape `[64,3,16,16]`, then flatten to groups for quantization and return to convolution layout. It evaluates the same candidate on ordinary random images and images with periodic contrast spikes.

This component test answers whether input distribution changes local error. It deliberately excludes transformer blocks, the language decoder, preprocessing, and task metrics, so its conclusion stops before full-model quality.

Only after these variables match the protocol should the cell be executed.


In [2]:
conv=torch.nn.Conv2d(3,64,kernel_size=16,stride=16,bias=False,device=device); w=conv.weight.detach(); _,_,dq=symmetric_quantize(w.reshape(64,-1),bits=4,group_size=192); dq=dq.reshape_as(w)
normal=torch.randn(8,3,224,224,device=device); contrast=normal.clone(); contrast[:,:,::16,::16]*=20
def project(x,weight): return torch.nn.functional.conv2d(x,weight,stride=16)
rows={}
for name,x in {"normal":normal,"high_contrast":contrast}.items(): rows[name]=error_metrics(project(x,w),project(x,dq))
result=base_result(21,"pytorch-gpu"); result.update({"patch_projection":{"weight_shape":list(w.shape),"group_size":192},"domain_errors":rows,
    "conclusion":"Patch-projection error changed with image distribution; no full VLM quality conclusion was made."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Patch weight shape | 64 × 3 × 16 × 16 |
| Normal-image RMSE | 0.040851 |
| High-contrast RMSE | 0.063011 |
| Normal max error | 0.202358 |
| High-contrast max error | 0.358466 |


## 7. Interpret rather than merely print

For normal images, projection RMSE was 0.040851 with cosine 0.997531. High-contrast patches increased RMSE to 0.063011 and max absolute error from 0.202358 to 0.358466, while cosine remained 0.997666.

The higher absolute error under contrast shift shows why one calibration distribution is insufficient even when cosine looks stable. Whether that change affects a VLM answer depends on downstream normalization and attention, which this lab does not model.

**Inspection rule:** Compare domain-specific errors and keep the experiment scoped to the patch projection, not an entire VLM quality claim.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The measured tensors and operations ran on CUDA through PyTorch. The result does not name a separate production backend unless an operator trace identifies it.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "Patch-projection error changed with image distribution; no full VLM quality conclusion was made.",
  "domain_errors": {
    "high_contrast": {
      "cosine": 0.99766618,
      "mae": 0.04951562,
      "max_abs": 0.35846561,
      "rmse": 0.06301098
    },
    "normal": {
      "cosine": 0.99753112,
      "mae": 0.03256194,
      "max_abs": 0.20235768,
      "rmse": 0.04085071
    }
  },
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:46:04+00:00",
  "lesson": 21,
  "patch_projection": {
    "group_size": 192,
    "weight_shape": [
      64,
      3,
      16,
      16
    ]
  },
  "schema_version": 1
}
Saved: artifacts/rtx5090-result.json


## 9. Make the bounded decision

> Calibrate and regress each modality and bridge component rather than applying a text-only decision globally.

**Acceptance/rollback:** Stratify calibration/evaluation by modality, resolution, prompt length, OCR/chart cases, and component; measure component error plus end-task multimodal quality.

**Failure analysis:** A text-only calibration set never exercises the patch projection. Average image embeddings can also hide OCR, diagrams, dark images, or saturated regions. Another mistake is to use image reconstruction metrics for a model whose deployment objective is answer correctness or grounding.


## 10. Extend the evidence

Capture activation ranges from photographs, documents, charts, OCR-heavy images, and high-contrast synthetic cases. Quantize vision encoder, connector, and decoder separately, then run end-to-end task slices. Use mixed precision when one modality component is consistently more sensitive.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
